# PROYECTO 1 · Cancelaciones hoteleras | CRISP-DM
**Versión integrada V2: trabajo del equipo + mejoras metodológicas de Juan**

**Pregunta:** ¿podemos ordenar las reservas por riesgo de cancelación con variables plausiblemente conocidas al reservar?

**Población principal:** se conservan las filas originales, incluidas las idénticas, porque no existe identificador único para asegurar que sean errores. No se altera `hotel_bookings.csv`.

**Evaluación principal:** particiones cronológicas por **fecha prevista de llegada**: entrenamiento antes del 01/07/2016, validación entre el 01/07/2016 y el 31/12/2016, prueba a partir del 01/01/2017. No equivale a una simulación histórica perfecta de cuándo se reservó ni cuándo se conocía el resultado.

**Variables:** mantenemos las 15 variables conservadoras del equipo. Comparamos regresión logística, árbol de decisión y, como **experimento adicional sujeto a autorización docente**, Random Forest. Se registran métricas, curva Precision–Recall, deciles/lift, auditoría y versiones. Las pruebas de deduplicación y de variables ampliadas no determinan el modelo final.

**Ejecución:** coloca `hotel_bookings.csv` junto al notebook o en la raíz del repositorio; ejecuta todas las celdas en orden. Las tablas de salida se almacenan en `resultados_integracion/`.


## Resumen ejecutivo · protocolo actualizado
- **Problema:** identificar reservas de mayor riesgo sin asumir que la predicción es causal.
- **Fases CRISP-DM:** negocio → comprensión de datos → preparación → modelado → evaluación → propuesta de despliegue.
- **Regla de decisión:** se selecciona el **modelo por Average Precision en validación** y el **umbral por F1 en validación**. La prueba temporal se consulta una sola vez, tras congelar ambas decisiones. La comparación aleatoria previa es histórica y **no es comparable directamente** con esta evaluación.
- **Decisión sobre calidad:** conservar duplicados en la población principal; cuantificar su efecto mediante un segundo experimento, sin seleccionar el escenario con mejor puntuación.
- **Transparencia:** el resultado de cada modelo debe consultarse en las salidas ejecutadas; no se copian las cifras de la versión anterior como si pertenecieran a este experimento.
- **Límite:** no conocemos una fotografía completa de variables al instante de la reserva ni los costos económicos de alertas falsas. No hay despliegue real.


# 1. Comprensión del negocio
### 1.1. Problema y decisión que queremos apoyar
Las cancelaciones reducen la certeza sobre la ocupación hotelera y pueden dificultar la planeación comercial. Se busca **identificar reservas de mayor riesgo desde su creación** para priorizar contactos o confirmaciones, no para penalizar automáticamente a huéspedes.

### 1.2. Objetivo general
Construir y evaluar un modelo básico de **clasificación supervisada binaria** para estimar la probabilidad de que una reserva termine cancelada o en no presentación, a partir de características que, razonablemente, se conocen al reservar.

### 1.3. Objetivos específicos
1. Describir reservas, cancelaciones y calidad del CSV original.
2. Explorar relaciones de la cancelación con anticipación, hotel, segmento, historial y otras dimensiones.
3. Preparar datos **sin utilizar información posterior a la creación de la reserva**.
4. Comparar una referencia simple, regresión logística y árbol de decisión con datos de validación.
5. Evaluar el modelo seleccionado en un conjunto de prueba no utilizado para elegirlo.
6. Examinar limitaciones mediante pruebas temporales y un escenario alternativo sin filas idénticas.
7. Explicar los resultados, riesgos y una propuesta de uso académico.

### 1.4. Cómo juzgaremos el resultado
- **Recall/sensibilidad:** proporción de reservas que efectivamente se cancelan y que el modelo detecta.
- **Precisión:** qué proporción de las reservas señaladas como riesgosas realmente se cancela.
- **F1:** equilibrio entre las dos anteriores; lo usaremos para **elegir modelo y umbral en validación**.
- **AUC-ROC:** capacidad de ordenar reservas de mayor a menor riesgo sin fijar un umbral.
- **Matriz de confusión:** falsos positivos y falsos negativos. Sin costos operativos disponibles, no podemos asegurar que el umbral de mejor F1 sea el más rentable para un hotel.

# 2. Comprensión de los datos y EDA
## 2.1. Entorno y lectura del CSV
**Qué hacemos:** importamos las librerías usadas en clase, fijamos una semilla para repetir los resultados y cargamos la fuente original sin modificarla.

In [ ]:
from pathlib import Path
import json, hashlib, platform, sys
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, roc_curve,
                             PrecisionRecallDisplay)

SEMILLA = 42
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 35)
actual = Path.cwd().resolve()
raices = [actual, *actual.parents]
ruta = next((p / "hotel_bookings.csv" for p in raices if (p / "hotel_bookings.csv").is_file()), None)
if ruta is None:
    try:
        from google.colab import files
        print("Sube hotel_bookings.csv")
        files.upload()
        ruta = Path("hotel_bookings.csv")
    except ImportError:
        raise FileNotFoundError("Ubica hotel_bookings.csv junto al notebook o en la raíz del repositorio")
if not ruta.is_file():
    raise FileNotFoundError("No se encontró hotel_bookings.csv")
RESULTADOS = Path.cwd() / "resultados_integracion"
RESULTADOS.mkdir(exist_ok=True, parents=True)
df = pd.read_csv(ruta, low_memory=False)
assert "is_canceled" in df.columns and df["is_canceled"].isin([0, 1]).all()
print("Filas:", len(df), "| Variables:", df.shape[1], "| Archivo:", ruta)
display(df.head(3))

**Interpretación:** una fila es una observación de reserva; varios registros pueden tener atributos idénticos sin representar necesariamente una duplicación errónea. No hay una clave inequívoca de reserva o de huésped.

In [ ]:
print("Tipos de variables:")
display(pd.DataFrame({"Tipo": df.dtypes.astype(str),
                      "Nulos": df.isna().sum(),
                      "Valores diferentes": df.nunique(dropna=False)}))
print("Resumen numérico (columnas más útiles para el problema):")
cols_describe = ["lead_time", "adr", "stays_in_weekend_nights", "stays_in_week_nights",
                 "adults", "children", "babies", "previous_cancellations", "total_of_special_requests"]
display(df[cols_describe].describe().T.round(2))

## 2.2. Calidad, consistencia y definición del target
**Qué hacemos:** contamos faltantes, filas idénticas y registros sospechosos; sin corregirlos todavía. Revisamos que `is_canceled` concuerde con el resultado registrado.

In [ ]:
nulos = df.isna().sum().loc[lambda x: x > 0].sort_values(ascending=False)
tabla_nulos = pd.DataFrame({"Faltantes": nulos,
                             "Porcentaje (%)": (100 * nulos / len(df)).round(2)})
display(tabla_nulos)

repetidas = int(df.duplicated().sum())
sin_huespedes = int(((df.adults + df.children.fillna(0) + df.babies) == 0).sum())
sin_noches = int(((df.stays_in_weekend_nights + df.stays_in_week_nights) == 0).sum())
print(f"Filas exactamente iguales a otra: {repetidas:,} ({repetidas/len(df):.2%})")
print("Filas sin huéspedes registrados:", sin_huespedes)
print("Filas sin noches de estadía:", sin_noches)
print("Tarifas diarias negativas:", int((df.adr < 0).sum()))
print("Tarifas diarias mayores a 1.000:", int((df.adr > 1000).sum()))

display(pd.crosstab(df["reservation_status"], df["is_canceled"], margins=True))
assert df.is_canceled.isin([0, 1]).all(), "Target con valores inesperados"
assert len(df) == 119390 and df.shape[1] == 32, "Verifica que cargaste el archivo esperado"

**Interpretación y decisión:** `is_canceled=1` incluye `Canceled` **y** `No-Show`; no equivale exclusivamente a cancelaciones explícitas. `company` está mayoritariamente vacío y `agent` tiene muchos faltantes, pero no podemos asumir automáticamente que un dato faltante equivale a «no aplica». Los registros idénticos **no se borran de la muestra principal**. Las tarifas extremas se documentan; `adr` no se utilizará en el modelo principal, por lo que no es necesario imputarla para predecir.

## 2.3. Anticipación, duración y tarifa: distribución univariada
**Qué hacemos:** observamos las formas y los valores extremos de tres variables numéricas relevantes. El histograma de `adr` limita el eje visual a valores de 0 a 500, pero **no elimina registros del CSV**.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
ejes[0].hist(df["lead_time"], bins=30)
ejes[0].set(title="Anticipación de la reserva", xlabel="Días (lead_time)", ylabel="Reservas")
noches = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
ejes[1].hist(noches, bins=range(0, 22))
ejes[1].set(title="Duración de la estancia", xlabel="Noches (0 a 20 mostradas)", ylabel="Reservas", xlim=(0, 20))
ejes[2].hist(df.loc[df.adr.between(0, 500), "adr"], bins=30)
ejes[2].set(title="Tarifa diaria: solo rango gráfico 0–500", xlabel="ADR", ylabel="Reservas")
plt.tight_layout(); plt.show()
print("Anticipación (media / mediana):", round(df.lead_time.mean(), 2), "/", df.lead_time.median())
print("Tarifa diaria (mín. / mediana / máx.):", df.adr.min(), "/", df.adr.median(), "/", df.adr.max())

**Interpretación:** una media mayor que la mediana de `lead_time` sugiere una cola hacia reservas con mayor anticipación. Las tarifas extremas merecen investigación de calidad; un valor llamativo no demuestra por sí solo que sea erróneo.

## 2.4. Distribución de la variable objetivo
**Qué hacemos:** comparamos cantidades y proporciones de reservas que terminaron canceladas o en *No-Show*.

In [ ]:
conteo_target = df.is_canceled.value_counts().sort_index()
tasa_general = float(df.is_canceled.mean())
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([0, 1], [conteo_target[0], conteo_target[1]])
ax.set(title=f"Resultado final de las reservas (tasa positiva {tasa_general:.1%})",
       xlabel="0 = No cancelada | 1 = Cancelada / No-Show", ylabel="Número de reservas")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout(); plt.show()
print("Tasa de resultado positivo:", f"{tasa_general:.2%}")

**Interpretación:** la clase positiva es minoritaria. Un modelo que siempre predice «no cancelada» ya tendría una exactitud aparentemente aceptable; por eso reportamos también sensibilidad, precisión, F1 y AUC-ROC.

## 2.5. Relaciones con la cancelación: anticipación y tipo de hotel
**Qué hacemos:** calculamos **tasas dentro de cada grupo** en lugar de confundir volumen con riesgo. Los intervalos de anticipación se crean para el EDA; el modelo utilizará `lead_time` original.

In [ ]:
bins = [-1, 7, 30, 90, 180, 365, float("inf")]
labels = ["0–7", "8–30", "31–90", "91–180", "181–365", "366+"]
df_eda = df.copy()
df_eda["rango_anticipacion"] = pd.cut(df_eda.lead_time, bins=bins, labels=labels)

def tabla_tasas(columna, datos=df_eda):
    tabla = (datos.groupby(columna, observed=True)["is_canceled"]
             .agg(Reservas="size", Canceladas="sum", Tasa="mean"))
    tabla["Tasa (%)"] = (100 * tabla.pop("Tasa")).round(2)
    return tabla

t_ant = tabla_tasas("rango_anticipacion")
t_hotel = tabla_tasas("hotel")
display(t_ant)
display(t_hotel)
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
t_ant["Tasa (%)"].plot(kind="bar", ax=ejes[0])
ejes[0].set(title="Cancelaciones por anticipación", xlabel="Días de anticipación", ylabel="Tasa de cancelación (%)")
t_hotel["Tasa (%)"].plot(kind="bar", ax=ejes[1])
ejes[1].set(title="Cancelaciones según hotel", xlabel="Tipo de hotel", ylabel="Tasa de cancelación (%)")
plt.tight_layout(); plt.show()

**Interpretación:** la tasa de cancelación sube desde aproximadamente **9,6 %** en reservas con 0–7 días de anticipación hasta cerca de **67,7 %** en las de más de 365 días. En el **City Hotel**, la tasa es **41,7 %**, frente a **27,8 %** en Resort Hotel. Esto describe asociaciones: no demuestra que anticipar o elegir un hotel sea la causa de la cancelación.

## 2.6. Relación con segmento comercial, recurrencia y tipo de cliente
**Qué hacemos:** analizamos dimensiones comerciales, tomando en cuenta cuántas reservas hay en cada grupo.

In [ ]:
for variable in ["market_segment", "distribution_channel", "customer_type", "is_repeated_guest"]:
    print("Variable:", variable)
    display(tabla_tasas(variable).sort_values("Reservas", ascending=False).head(10))

segmentos = tabla_tasas("market_segment").sort_values("Reservas", ascending=False).head(6)
fig, ax = plt.subplots(figsize=(9, 4))
segmentos["Tasa (%)"].sort_values().plot(kind="barh", ax=ax)
ax.set(title="Tasa de cancelación por segmento (6 más numerosos)",
       xlabel="Cancelaciones en cada segmento (%)", ylabel="Segmento")
plt.tight_layout(); plt.show()

**Interpretación:** el segmento **Groups** presenta aproximadamente **61,1 %** de cancelaciones, frente a **15,3 %** del segmento **Direct**. Las reservas de huéspedes recurrentes presentan **14,5 %** frente a **37,8 %** entre no recurrentes. Son diferencias descriptivas; puede haber influencia de otras características como la anticipación y el hotel.

## 2.7. Historial, solicitudes y depósito: señales que exigen cautela
**Qué hacemos:** exploramos asociaciones del historial de cancelaciones y de solicitudes especiales. También observamos la variable depósito, **solo como contexto**, porque no está acreditado que refleje el estado inicial de la reserva.

In [ ]:
df_eda["tiene_cancelacion_previa"] = (df_eda.previous_cancellations > 0).map({False: "No", True: "Sí"})
for variable in ["tiene_cancelacion_previa", "total_of_special_requests", "deposit_type"]:
    print("Variable:", variable)
    display(tabla_tasas(variable).sort_values("Reservas", ascending=False).head(10))

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
tabla_tasas("tiene_cancelacion_previa")["Tasa (%)"].plot(kind="bar", ax=ejes[0])
ejes[0].set(title="Historial de cancelaciones", xlabel="¿Hay cancelaciones previas?", ylabel="Tasa (%)")
tabla_tasas("total_of_special_requests")["Tasa (%)"].plot(kind="bar", ax=ejes[1])
ejes[1].set(title="Solicitudes especiales (EDA, no predictor)", xlabel="Número de solicitudes", ylabel="Tasa (%)")
plt.tight_layout(); plt.show()

**Interpretación:** un historial de cancelaciones anteriores puede ser útil si fue construido con hechos previos a la reserva. El número de solicitudes especiales y el depósito muestran asociaciones muy fuertes (en `Non Refund`, casi **99,4 %**), pero **no entran al modelo**: el dataset no demuestra que sus valores sean los de la fecha de creación. No hay fundamento para interpretar ese 99,4 % como efecto causal de una política no reembolsable.

## 2.8. País de origen y tendencias temporales
**Qué hacemos:** analizamos `country` y la evolución mensual de tasas, pero no usamos país como predictor principal porque su disponibilidad en la creación de la reserva no está demostrada. La cobertura de 2015 y 2017 es parcial.

In [ ]:
df_eda["country_eda"] = df_eda["country"].fillna("Sin dato")
paises = df_eda["country_eda"].value_counts().head(8).index
t_paises = tabla_tasas("country_eda").loc[paises]
display(t_paises)

fecha_llegada = pd.to_datetime(
    df.arrival_date_year.astype(str) + "-" + df.arrival_date_month + "-" +
    df.arrival_date_day_of_month.astype(str), format="%Y-%B-%d", errors="coerce")
assert fecha_llegada.notna().all(), "Hay fechas de llegada inválidas"
df_eda["mes_llegada"] = fecha_llegada.dt.to_period("M").astype(str)
serie_mes = df_eda.groupby("mes_llegada")["is_canceled"].agg(Reservas="size", Tasa="mean")

fig, ejes = plt.subplots(1, 2, figsize=(14, 4))
t_paises["Tasa (%)"].sort_values().plot(kind="barh", ax=ejes[0])
ejes[0].set(title="Países más frecuentes (EDA)", xlabel="Tasa de cancelación (%)", ylabel="País")
(serie_mes.Tasa * 100).plot(kind="line", marker="o", ax=ejes[1])
ejes[1].tick_params(axis="x", rotation=60)
ejes[1].set(title="Evolución por mes de llegada", xlabel="Mes de llegada", ylabel="Tasa de cancelación (%)")
plt.tight_layout(); plt.show()
print("Primera llegada:", fecha_llegada.min().date(), "| Última:", fecha_llegada.max().date())

**Interpretación:** los cambios mensuales son descriptivos y no prueban cambios causales de comportamiento. **2015 solo contiene julio–diciembre y 2017 enero–agosto**, por lo que la comparación de años completos estaría sesgada por periodos diferentes.

## 2.9. Correlaciones numéricas
**Qué hacemos:** calculamos correlaciones de Pearson, vistas en clase, solo entre variables numéricas interpretables. Identificadores como `agent` o `company` **no deben tratarse como magnitudes continuas**.

In [ ]:
columnas_corr = ["is_canceled", "lead_time", "stays_in_weekend_nights",
                  "stays_in_week_nights", "adults", "children", "babies",
                  "is_repeated_guest", "previous_cancellations",
                  "previous_bookings_not_canceled", "adr", "total_of_special_requests"]
correlacion = df[columnas_corr].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlacion, cmap="coolwarm", center=0, annot=True, fmt=".2f", ax=ax)
ax.set_title("Correlación entre variables numéricas (datos originales)")
plt.tight_layout(); plt.show()
display(correlacion["is_canceled"].drop("is_canceled").sort_values(ascending=False).to_frame("Correlación con target").round(3))

### Conclusiones de comprensión de datos
- `lead_time` y algunos atributos comerciales muestran asociaciones con `is_canceled`.
- Las relaciones no son causales y las correlaciones numéricas no sustituyen una evaluación predictiva.
- Hay faltantes, filas idénticas, reservas sin huéspedes o noches y valores extremos; **no se deben corregir por intuición**.
- El target reúne cancelaciones y *No-Show*.
- **Prohibición metodológica:** `reservation_status` y `reservation_status_date` revelan el desenlace; `assigned_room_type`, `booking_changes` y otras variables actualizadas posteriormente no deben entrar al modelo de predicción al crear la reserva.

Pasamos a preparación sin eliminar registros ni redefinir el target.

# 3. Preparación de los datos · escenario principal

**Qué mantenemos:** las 15 variables conservadoras del equipo. La decisión de no incluir ADR, depósito, país o cambios posteriores no significa que sean inútiles; significa que **no está demostrada** su disponibilidad y estabilidad al momento de predecir.

**Qué incorporamos de Juan:** auditoría de calidad, trazabilidad por fila, separación temporal, métricas complementarias y registro de ejecución. Conservar filas idénticas evita decidir sin un identificador confiable si una coincidencia es un error o una reserva legítima. El caso deduplicado se analiza por separado.

In [ ]:
columnas_num = ["lead_time", "stays_in_weekend_nights", "stays_in_week_nights", "adults", "children",
                "babies", "is_repeated_guest", "previous_cancellations", "previous_bookings_not_canceled"]
columnas_cat = ["hotel", "arrival_date_month", "meal", "market_segment",
                "distribution_channel", "reserved_room_type"]
columnas_modelo = columnas_num + columnas_cat
columnas_excluidas = ["reservation_status", "reservation_status_date", "assigned_room_type",
                     "booking_changes", "days_in_waiting_list", "deposit_type", "adr",
                     "total_of_special_requests", "required_car_parking_spaces", "country",
                     "agent", "company", "customer_type", "arrival_date_year"]
assert len(columnas_modelo) == 15 and not set(columnas_modelo) & set(columnas_excluidas)
requeridas = set(columnas_modelo + ["is_canceled", "arrival_date_year", "arrival_date_day_of_month"])
faltan = requeridas - set(df.columns)
if faltan: raise ValueError(f"Faltan columnas: {sorted(faltan)}")
# ID técnico de la fila del CSV: no identifica una reserva ni un cliente real.
base = df.copy()
base["fila_csv"] = np.arange(len(base)) + 2
meses = {m:i for i,m in enumerate(["January","February","March","April","May","June",
                                   "July","August","September","October","November","December"], 1)}
base["fecha_llegada"] = pd.to_datetime(pd.DataFrame({
    "year": base.arrival_date_year,
    "month": base.arrival_date_month.map(meses),
    "day": base.arrival_date_day_of_month}), errors="coerce")
if base.fecha_llegada.isna().any():
    raise ValueError("Hay fechas inválidas; no se puede hacer partición temporal sin resolverlas")
for col in columnas_cat:
    base[col] = base[col].astype("string").astype(object).where(base[col].notna(), np.nan)

calidad = pd.DataFrame({"Variable":df.columns, "Tipo":df.dtypes.astype(str).values,
                        "Nulos":df.isna().sum().values,
                        "Porcentaje_nulos":(df.isna().mean().values * 100).round(2),
                        "Valores_distintos":df.nunique(dropna=True).values})
calidad.to_csv(RESULTADOS / "auditoria_columnas.csv", index=False, encoding="utf-8-sig")
limpieza = pd.DataFrame([
    {"Decisión":"Archivo original", "Filas":len(df), "Eliminadas":0,
     "Justificación":"No se altera el archivo fuente"},
    {"Decisión":"Conservar filas completamente idénticas", "Filas":len(base), "Eliminadas":0,
     "Justificación":f"{int(df.duplicated().sum())} duplicados exactos sin identificador único"},
    {"Decisión":"Imputación dentro del pipeline", "Filas":len(base), "Eliminadas":0,
     "Justificación":"Las medianas y modas se calculan SOLO en entrenamiento"}
])
limpieza.to_csv(RESULTADOS / "registro_limpieza.csv", index=False, encoding="utf-8-sig")
print("Registros:",len(base),"| Idénticos adicionales:",int(df.duplicated().sum()),
      "| Cancelación:",f"{base.is_canceled.mean():.2%}")
display(limpieza)
display(calidad.sort_values("Porcentaje_nulos",ascending=False).head(12))

## 3.2. División temporal: entrenamiento, validación y prueba

- **Entrenamiento:** fechas de llegada anteriores al 1 de julio de 2016.
- **Validación:** julio a diciembre de 2016; sirve para seleccionar modelo y umbral.
- **Prueba:** llegadas desde enero de 2017; permanece intacta hasta completar la selección.

Esta separación es más realista para evaluar periodos futuros que una aleatoria, pero **fecha de llegada no equivale a fecha de creación de la reserva**. Tampoco tenemos la fecha exacta en que se conocía cada etiqueta; no afirmamos ausencia completa de fuga temporal.

In [ ]:
train = base.loc[base.fecha_llegada < "2016-07-01"].copy()
valid = base.loc[(base.fecha_llegada >= "2016-07-01") & (base.fecha_llegada < "2017-01-01")].copy()
test = base.loc[base.fecha_llegada >= "2017-01-01"].copy()
for nombre, parte in [("entrenamiento",train),("validacion",valid),("prueba",test)]:
    if parte.empty or parte.is_canceled.nunique()!=2: raise ValueError(f"La partición {nombre} no es válida")
assert train.fecha_llegada.max() < valid.fecha_llegada.min() < test.fecha_llegada.min()
assert len(train)+len(valid)+len(test)==len(base)
X_train,y_train = train[columnas_modelo],train.is_canceled.astype(int)
X_val,y_val = valid[columnas_modelo],valid.is_canceled.astype(int)
X_test,y_test = test[columnas_modelo],test.is_canceled.astype(int)
particiones = pd.DataFrame([{"Conjunto":nombre,"Reservas":len(parte),
                             "Desde":str(parte.fecha_llegada.min().date()),
                             "Hasta":str(parte.fecha_llegada.max().date()),
                             "Cancelacion":float(parte.is_canceled.mean())}
                            for nombre,parte in [("entrenamiento",train),("validacion",valid),("prueba",test)]])
particiones.to_csv(RESULTADOS/"particiones.csv",index=False,encoding="utf-8-sig")
asignacion = pd.concat([parte[["fila_csv","fecha_llegada"]].assign(conjunto=nombre)
                        for nombre,parte in [("train",train),("validacion",valid),("test",test)]])
asignacion.to_csv(RESULTADOS/"asignacion_particiones.csv",index=False,encoding="utf-8-sig")
display(particiones)
print("La fila_csv es una posición del CSV, no un identificador único de reserva.")

## 3.3. Transformaciones sin fuga desde validación o prueba

**Por qué pipeline:** medianas, moda, escalamiento y categorías se aprenden únicamente de entrenamiento; las mismas reglas se aplican luego a validación y prueba. Las tres familias de modelos reciben **las mismas 15 variables y exactamente las mismas filas**.

In [ ]:
def crear_pipeline(tipo):
    prep_num = Pipeline([("mediana", SimpleImputer(strategy="median")),
                         ("escala", StandardScaler())])
    prep_cat = Pipeline([("moda", SimpleImputer(strategy="most_frequent")),
                         ("columnas_01", OneHotEncoder(handle_unknown="ignore"))])
    preparador = ColumnTransformer([("numericas",prep_num,columnas_num),
                                     ("categoricas",prep_cat,columnas_cat)])
    if tipo == "Regresión logística":
        estimador = LogisticRegression(max_iter=1500,random_state=SEMILLA)
    elif tipo == "Árbol de decisión":
        estimador = DecisionTreeClassifier(max_depth=6,min_samples_leaf=100,random_state=SEMILLA)
    elif tipo == "Random Forest":
        estimador = RandomForestClassifier(n_estimators=150,max_depth=14,
                                           min_samples_leaf=10,n_jobs=2,random_state=SEMILLA)
    else: raise ValueError("Modelo no contemplado")
    return Pipeline([("preparacion",preparador),("modelo",estimador)])
modelos = ["Regresión logística","Árbol de decisión","Random Forest"]
print("Candidatos (Random Forest sujeto a autorización docente):",modelos)

# 4. Modelado

**Selección predefinida:** primero elegimos el modelo con mayor **Average Precision (AP) en validación**, usando ROC-AUC como criterio de desempate. Luego elegimos **dentro del modelo ganador** el umbral que maximiza F1 en validación. Esto evita utilizar test para ajustar decisiones.

AP resume precisión-recall a lo largo de los umbrales; su referencia aproximada es la proporción de canceladas. F1 depende del umbral y sirve para equilibrar precisión y recall, pero **no representa un óptimo económico**.

In [ ]:
def obtener_metricas(nombre,reales,probas,umbral=0.5):
    pred=(np.asarray(probas)>=umbral).astype(int)
    return {"Modelo":nombre,"Umbral":float(umbral),
            "Exactitud":float(accuracy_score(reales,pred)),
            "Precisión":float(precision_score(reales,pred,zero_division=0)),
            "Recall":float(recall_score(reales,pred,zero_division=0)),
            "F1":float(f1_score(reales,pred,zero_division=0)),
            "AUC-ROC":float(roc_auc_score(reales,probas)),
            "Average Precision":float(average_precision_score(reales,probas)),
            "Porcentaje alertas":float(pred.mean())}
modelos_entrenados={}
prob_val={}
filas_validacion=[]
umbrales = np.arange(.10,.901,.025)
for nombre in modelos:
    print("Entrenando:",nombre)
    p=crear_pipeline(nombre).fit(X_train,y_train)
    pv=p.predict_proba(X_val)[:,1]
    modelos_entrenados[nombre]=p
    prob_val[nombre]=pv
    for t in umbrales:
        filas_validacion.append(obtener_metricas(nombre,y_val,pv,float(t)))
resultados_validacion=pd.DataFrame(filas_validacion)
mejores_por_modelo=(resultados_validacion.sort_values(["Modelo","F1","Umbral"],
                            ascending=[True,False,False]).drop_duplicates("Modelo"))
comparacion=(mejores_por_modelo.sort_values(["Average Precision","AUC-ROC","Modelo"],
                                           ascending=[False,False,True]).reset_index(drop=True))
modelo_elegido=str(comparacion.iloc[0]["Modelo"])
umbral_elegido=float(comparacion.iloc[0]["Umbral"])
comparacion.to_csv(RESULTADOS/"comparacion_validacion.csv",index=False,encoding="utf-8-sig")
resultados_validacion.to_csv(RESULTADOS/"umbrales_validacion.csv",index=False,encoding="utf-8-sig")
display(comparacion.round(4))
print("Modelo congelado:",modelo_elegido,"| Umbral congelado:",round(umbral_elegido,3))
fig,ax=plt.subplots(figsize=(9,4))
for nombre in modelos:
    t=resultados_validacion.query("Modelo == @nombre")
    ax.plot(t["Umbral"],t["F1"],label=nombre)
ax.axvline(umbral_elegido,color="black",ls="--",label="Umbral elegido")
ax.set(xlabel="Umbral",ylabel="F1 en validación",title="Elección del umbral exclusivamente en validación")
ax.legend(); plt.tight_layout(); plt.show()

# 5. Evaluación temporal final

**Regla:** el conjunto de prueba de 2017 se consulta **una sola vez**, después de elegir modelo y umbral. Reportamos matriz de confusión, ROC-AUC, AP, precisión, recall, F1 y porcentaje de alertas. No modificamos el modelo usando esos resultados.

In [ ]:
modelo_final=modelos_entrenados[modelo_elegido]
proba_test=modelo_final.predict_proba(X_test)[:,1]
pred_test=(proba_test>=umbral_elegido).astype(int)
resultado_test=obtener_metricas(modelo_elegido,y_test,proba_test,umbral_elegido)
pd.DataFrame([resultado_test]).to_csv(RESULTADOS/"evaluacion_test.csv",index=False,encoding="utf-8-sig")
display(pd.DataFrame([resultado_test]).round(4))
matriz=confusion_matrix(y_test,pred_test,labels=[0,1])
tn,fp,fn,tp=matriz.ravel()
print(f"TN={tn:,} | FP={fp:,} | FN={fn:,} | TP={tp:,} | Prevalencia={y_test.mean():.2%}")
fig,ejes=plt.subplots(1,3,figsize=(16,4.5))
sns.heatmap(matriz,annot=True,fmt=",d",cmap="Blues",cbar=False,ax=ejes[0],
            xticklabels=["No cancelada","Cancelada"],yticklabels=["No cancelada","Cancelada"])
ejes[0].set(title="Matriz de confusión · test temporal",xlabel="Predicción",ylabel="Real")
fpr,tpr,_=roc_curve(y_test,proba_test)
ejes[1].plot(fpr,tpr,label=f"AUC={resultado_test['AUC-ROC']:.3f}")
ejes[1].plot([0,1],[0,1],"--",color="gray")
ejes[1].set(title="Curva ROC · test temporal",xlabel="Falsos positivos",ylabel="Verdaderos positivos")
ejes[1].legend()
PrecisionRecallDisplay.from_predictions(y_test,proba_test,ax=ejes[2])
ejes[2].axhline(y_test.mean(),color="gray",ls="--",label="Prevalencia")
ejes[2].set_title("Precision–Recall · test temporal")
ejes[2].legend()
plt.tight_layout();plt.savefig(RESULTADOS/"evaluacion_test.png",dpi=140,bbox_inches="tight");plt.show()

## 5.2. Priorización de reservas: deciles y lift

Ordenamos las reservas de prueba por riesgo predicho. **Decil 1** es el 10 % con puntuación más alta. El *lift* divide la tasa de cancelación de un decil por la tasa general del conjunto de prueba. Un lift de 2 significa el doble de concentración que en el promedio; **no garantiza causalidad ni precisión probabilística**.

In [ ]:
ranking=pd.DataFrame({"fila_csv":test.fila_csv.to_numpy(),"fecha_llegada":test.fecha_llegada.to_numpy(),
                     "Real":y_test.to_numpy(),"Riesgo estimado":proba_test,
                     "Alerta":pred_test}).sort_values(["Riesgo estimado","fila_csv"],ascending=[False,True]).reset_index(drop=True)
# Clasificar por posición (no por qcut de probabilidades repetidas).
ranking["Decil"]=(np.floor(np.arange(len(ranking))*10/len(ranking)).astype(int)+1)
deciles=(ranking.groupby("Decil").agg(Reservas=("Real","size"),Canceladas=("Real","sum"),
                                     Tasa_cancelacion=("Real","mean")).reset_index())
deciles["Lift"]=deciles.Tasa_cancelacion/y_test.mean()
deciles["Proporcion_cancelaciones"]=deciles.Canceladas/y_test.sum()
ranking.to_csv(RESULTADOS/"predicciones_test.csv",index=False,encoding="utf-8-sig")
deciles.to_csv(RESULTADOS/"deciles_test.csv",index=False,encoding="utf-8-sig")
display(deciles.round(4))
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(deciles.Decil,100*deciles.Tasa_cancelacion)
ax.axhline(100*y_test.mean(),linestyle="--",color="gray",label="Tasa media en test")
ax.set(xlabel="Decil de riesgo (1 = más alto)",ylabel="Tasa de cancelación (%)",
       title="Concentración observada de cancelaciones por decil")
ax.set_xticks(range(1,11));ax.legend();plt.tight_layout();plt.show()
print("Lift del decil 1:",round(float(deciles.loc[0,"Lift"]),2))

## 5.3. Variables destacadas y límites

Mostramos importancia de atributos transformados en el modelo seleccionado. Los coeficientes y las importancias son asociaciones **del modelo**, no explicaciones causales. La codificación one-hot crea varias columnas por variable original.

In [ ]:
nombres_transformados=modelo_final.named_steps["preparacion"].get_feature_names_out()
interno=modelo_final.named_steps["modelo"]
if modelo_elegido=="Regresión logística":
    pesos=pd.Series(interno.coef_[0],index=nombres_transformados)
    relevantes=pesos.reindex(pesos.abs().nlargest(12).index)
    etiqueta="Coeficiente (con signo)"
else:
    pesos=pd.Series(interno.feature_importances_,index=nombres_transformados)
    relevantes=pesos.nlargest(12)
    etiqueta="Importancia interna del modelo"
display(relevantes.to_frame(etiqueta).round(3))
fig,ax=plt.subplots(figsize=(9,4.5))
relevantes.sort_values().plot.barh(ax=ax)
ax.set(xlabel=etiqueta,title="Atributos destacados por el modelo")
plt.tight_layout();plt.show()

# 6. Robustez · análisis de sensibilidad (no el criterio de selección)

**Escenario alternativo:** eliminar filas completamente idénticas y ejecutar de nuevo el mismo protocolo temporal, con las mismas 15 variables y las mismas reglas para elegir modelo y umbral. La comparación muestra sensibilidad a una decisión de calidad, **no cuál población es la verdadera**. Esta prueba no debe utilizarse para ajustar el modelo principal después de consultar su test.

In [ ]:
sin_duplicados=base.loc[~df.duplicated()].copy()
print("Principal:",len(base),"filas; cancelación:",f"{base.is_canceled.mean():.2%}")
print("Sin idénticos:",len(sin_duplicados),"filas; cancelación:",f"{sin_duplicados.is_canceled.mean():.2%}")
s_train=sin_duplicados.loc[sin_duplicados.fecha_llegada<"2016-07-01"]
s_val=sin_duplicados.loc[(sin_duplicados.fecha_llegada>="2016-07-01") &
                            (sin_duplicados.fecha_llegada<"2017-01-01")]
s_test=sin_duplicados.loc[sin_duplicados.fecha_llegada>="2017-01-01"]
if min(len(s_train),len(s_val),len(s_test))==0:raise ValueError("Particiones vacías en sensibilidad")
filas_s=[]
modelos_s={}
for nombre in modelos:
    p=crear_pipeline(nombre).fit(s_train[columnas_modelo],s_train.is_canceled)
    modelos_s[nombre]=p
    pv=p.predict_proba(s_val[columnas_modelo])[:,1]
    for t in umbrales:
        filas_s.append(obtener_metricas(nombre,s_val.is_canceled,pv,float(t)))
res_s=pd.DataFrame(filas_s)
por_modelo_s=res_s.sort_values(["Modelo","F1","Umbral"],ascending=[True,False,False]).drop_duplicates("Modelo")
mejor_s=por_modelo_s.sort_values(["Average Precision","AUC-ROC","Modelo"],ascending=[False,False,True]).iloc[0]
nombre_s=str(mejor_s.Modelo); umbral_s=float(mejor_s.Umbral)
prob_s=modelos_s[nombre_s].predict_proba(s_test[columnas_modelo])[:,1]
resultado_s=obtener_metricas(nombre_s,s_test.is_canceled,prob_s,umbral_s)
comparacion_escenarios=pd.DataFrame([
    {"Escenario":"Con filas idénticas","Filas":len(base),"Cancelacion":base.is_canceled.mean(),**resultado_test},
    {"Escenario":"Sin filas idénticas","Filas":len(sin_duplicados),"Cancelacion":sin_duplicados.is_canceled.mean(),**resultado_s}])
comparacion_escenarios.to_csv(RESULTADOS/"sensibilidad_duplicados.csv",index=False,encoding="utf-8-sig")
display(comparacion_escenarios.round(4))
print("ADVERTENCIA: distintas poblaciones/test; métricas NO comparables como torneo de modelos.")

## 6.2. Solapamiento de perfiles entre periodos

Se cuentan filas de prueba cuyos **15 predictores** coinciden exactamente con algún perfil de entrenamiento. Un perfil idéntico **no prueba** que sea la misma reserva. Se reporta solo como advertencia sobre estructuras repetidas.

In [ ]:
firma_train=pd.util.hash_pandas_object(X_train,index=False).to_numpy()
firma_test=pd.util.hash_pandas_object(X_test,index=False).to_numpy()
coincidencias=np.isin(firma_test,np.unique(firma_train))
print(f"Perfiles de test coincidentes con entrenamiento: {coincidencias.sum():,} / {len(coincidencias):,} ({coincidencias.mean():.2%})")

# 7. Reproducibilidad, conclusiones y propuesta de despliegue

**Trazabilidad:** guardamos la suma SHA-256 de `hotel_bookings.csv`, versiones de librerías, parámetros, fechas, métricas y particiones. El directorio de resultados es una salida derivada, no se recomienda subir archivos con datos de clientes reales.

In [ ]:
paquetes=["pandas","numpy","scikit-learn","matplotlib","seaborn"]
versiones={}
for paquete in paquetes:
    try: versiones[paquete]=version(paquete)
    except PackageNotFoundError: versiones[paquete]="no detectado"
hash_csv=hashlib.sha256(ruta.read_bytes()).hexdigest()
metadata={"proyecto":"Proyecto 1 - CRISP-DM integrado V2", "fecha_utc":datetime.now(timezone.utc).isoformat(),
          "python":platform.python_version(), "versiones":versiones,"semilla":SEMILLA,
          "csv":str(ruta),"sha256_csv":hash_csv,"filas_originales":int(len(df)),
          "duplicados_exactos":int(df.duplicated().sum()),"variables":columnas_modelo,
          "particion":"fecha de llegada: train < 2016-07-01; val < 2017-01-01; test >= 2017-01-01",
          "regla_modelo":"Mayor Average Precision en validación; desempate AUC-ROC",
          "regla_umbral":"Máximo F1 en validación; empate a favor del umbral mayor",
          "modelo_elegido":modelo_elegido,"umbral":float(umbral_elegido),
          "resultado_test":resultado_test,
          "limites":["Fecha de llegada no es fecha de reserva", "No hay snapshot de variables iniciales",
                     "No hay costos ni intervención real", "Random Forest sujeto a autorización docente"]}
(RESULTADOS/"metadata_ejecucion.json").write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")
print("SHA-256 del CSV:",hash_csv)
print("Resultados guardados en:",RESULTADOS.resolve())
print("Modelo final:",modelo_elegido,"| Umbral:",umbral_elegido)
display(pd.DataFrame([resultado_test]).round(4))

## 7.2. Interpretación y acciones propuestas (no implementadas)

- **Qué hicimos:** conservar el EDA del equipo; elegir entre tres modelos con las mismas observaciones, variables y particiones temporales; congelar decisiones antes del test; medir AP, AUC, precisión, recall, F1, alertas y lift; auditar calidad y reproducibilidad.
- **Qué no podemos asegurar:** que una fila idéntica sea duplicación errónea; que todas las variables reflejen el momento de reservar; que el modelo ya esté validado comercialmente; que una puntuación individual sea una probabilidad calibrada.
- **Propuesta de despliegue académico:** registrar solo variables disponibles al reservar, confirmar con el hotel su estabilidad, generar ranking de riesgo, permitir que un auxiliar priorice seguimientos voluntarios y medir respuestas/cancelaciones sin castigar automáticamente al huésped.
- **Piloto sugerido:** dueño del proceso = responsable de reservas; métricas = precisión de alertas, recall (cuando se conozcan resultados), tasa de contacto, costo por contacto, cancelaciones y reclamaciones; revisar por mes/hotel, cambios de distribución y errores; pausar ante fallas de calidad de datos.
- **Decisiones pendientes con el profesor:** autorizar Random Forest y Average Precision; aprobar usar fecha de llegada como aproximación temporal; determinar si la fase 6 exige una implementación o basta con plan de despliegue; pedir orientación para justificar duplicados.

**Nota:** este experimento conserva los datos reales; no crea conclusiones numéricas estáticas porque cada persona debe verificarlas al ejecutar el cuaderno. No reentrenamos después de mirar el test.